# Category evidence by operation and valence

Simple, direct version: load the full `decoding_results.csv` dataframe (raw,
one row per decoded TR), add a few columns to make filtering easy, then use
plain dataframe filters + seaborn to plot. No abstraction layer -- every
plot below is a short, self-contained function you can copy and adapt.

For the item/category classifier (`regressor_label` -- whichever of
face/place/item this classifier actually decodes; see below): each row's
own **self-evidence** (`evidence_<its own true category>`) is the value
plotted throughout -- "how much did the classifier believe this trial was
its own true category".

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # run from _interactive_notebooks/, repo root one level up
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from workflows.generate_report import resolve_desc, list_subject_dirs, parse_subjects_arg, subject_paths

%matplotlib inline
sns.set_theme(style="ticks")

## Configuration -- edit these

In [ ]:
# -- where the data is -- two options:
# (1) a single already-merged CSV (e.g. generate_report.py's own
#     <desc>_group_decoding_results.csv, or any decoding_results.csv) --
#     set GROUP_CSV_PATH and leave the rest alone.
# (2) load + concatenate each subject's own decoding_results.csv straight
#     from the cluster's output tree -- leave GROUP_CSV_PATH = None and set
#     ANALYSIS_OUTPUT_DIR/DESC/SUBJECTS instead.
GROUP_CSV_PATH = None  # e.g. "/path/to/vvps_category_loc2WM_timecourse_classifier_group_decoding_results.csv"

ANALYSIS_OUTPUT_DIR = "/path/to/analysis/output"
DESC = "vvps_category_loc2WM_timecourse_classifier"
CONFIG_PATH = None      # optional: read DESC from a config's model.desc instead
SUBJECTS = None         # comma-separated list, a subject-list.txt path, or None for everyone

# per-operation colors -- matching this project's established color coding
OPERATION_COLORS = {"maintain": "green", "switch": "blue", "suppress": "red", "clear": "orange"}
VALENCE_COLORS = {"pos": "red", "neg": "blue"}

TR_SECONDS = 0.460     # seconds per TR -- x-axis everywhere below is window_index * TR_SECONDS
TRIAL_ONSET_SEC = 2.0  # time (seconds from window start) the operation/trial begins -- match this
                       # to model_conditions.timecourse_decoding.window's own onset offset
TRIAL_OFFSET_SEC = 6.0  # time (seconds from window start) the operation/trial ends -- adjust to
                        # your actual trial duration (onset + duration); this is a placeholder

OUTPUT_DIR = "./category_evidence_analysis" 

## Load data + add a few columns for easy filtering

In [ ]:
if GROUP_CSV_PATH:
    df = pd.read_csv(GROUP_CSV_PATH, dtype={"subject": str}, keep_default_na=True, na_values=[""])
    desc = DESC
    print(f"Loaded {len(df)} rows from {GROUP_CSV_PATH} -- {df['subject'].nunique()} subject(s)")
else:
    desc = resolve_desc(None, CONFIG_PATH) if CONFIG_PATH else DESC
    subjects_arg = parse_subjects_arg(SUBJECTS) if SUBJECTS else None
    subjects = list_subject_dirs(ANALYSIS_OUTPUT_DIR, desc, subjects=subjects_arg)
    frames = []
    for s in subjects:
        path = subject_paths(ANALYSIS_OUTPUT_DIR, desc, s)["decoding_raw"]
        if os.path.exists(path):
            frames.append(pd.read_csv(path, dtype={"subject": str}, keep_default_na=True, na_values=[""]))
    df = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(df)} rows from {len(subjects)} subject(s)")

# continuous timecourse decoding leaves regressor_label blank on real per-TR
# rows that aren't a scored category -- e.g. view_face/probe/fixation frames
# (see README.md section 4). keep_default_na/na_values above make sure a
# blank CSV field reads back as a real NaN rather than relying on pandas'
# default silently; replace() is a second, cheap safety net catching any
# literal empty string that slips through either read (this project has
# already hit one pandas-version-specific surprise with how a real missing
# value shows up in a "str"-dtype column, in evaluate_query_node).
df = df.replace(r"^\s*$", pd.NA, regex=True)

# -- a few extra columns to make filtering/sorting simple --
df["valence"] = df["task"].map({"WMpos": "pos", "WMneg": "neg"})           # pos / neg
df["object"] = df["regressor_label"]                                       # face / place / item (whichever this classifier decodes) -- NaN on unscored frames
df["operation"] = df["trial_type"].str.extract(r"(maintain|suppress|switch|clear)", expand=False)
df["time_sec"] = df["window_index"] * TR_SECONDS                           # x-axis for every plot below

# self-evidence: this row's own evidence_<its own true "object"> column --
# only defined where "object" is; every frame stays in the dataframe either
# way (still usable for plots/filters keyed on trial_type directly), this
# just leaves "value" NaN on frames that were decoded but never scored,
# rather than crashing on a literal "evidence_nan" column lookup
df["value"] = df.apply(lambda r: r[f"evidence_{r['object']}"] if pd.notna(r["object"]) else pd.NA, axis=1)

CATEGORIES = sorted(df["object"].dropna().unique())      # whichever of face/place/item this classifier actually decodes
OPERATIONS = [op for op in OPERATION_COLORS if op in df["operation"].unique()]
print("categories found:", CATEGORIES)
print("operations found:", OPERATIONS)
df[["subject", "object", "operation", "valence", "time_sec", "value"]].head()

## Plot 1: category evidence by operation, one figure per valence

3 subplots (one per category found above), each overlaying every operation
(`OPERATION_COLORS`) -- separate figures for pos and neg trials.

In [ ]:
def plot_categories_by_operation(data, title):
    # one row per (subject, category, operation, time) -- so seaborn's CI
    # band reflects between-subject variability, not raw single-trial noise
    per_subject = data.groupby(["subject", "object", "operation", "time_sec"])["value"].mean().reset_index()

    fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(5 * len(CATEGORIES), 4.5), sharex=True, sharey=True)
    for ax, cat in zip(axes, CATEGORIES):
        sns.lineplot(data=per_subject[per_subject["object"] == cat], x="time_sec", y="value",
                     hue="operation", hue_order=OPERATIONS, palette=OPERATION_COLORS, ax=ax)
        ax.axvline(TRIAL_ONSET_SEC, color="gray", linestyle="--", linewidth=0.75)
        ax.axvline(TRIAL_OFFSET_SEC, color="gray", linestyle=":", linewidth=0.75)
        ax.set_title(cat)
        ax.set_xlabel("Time (s)")
    axes[0].set_ylabel("classifier evidence (own true category)")
    fig.suptitle(title, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    return fig


for valence in ("pos", "neg"):
    plot_categories_by_operation(df[df["valence"] == valence], f"{desc}: category evidence by operation -- {valence}")
    plt.show()

## Plot 2: pos vs. neg, one figure per operation

3 subplots per figure: each category found above, plus "combined" (face +
place averaged together -- valence only applies to face/place trials in
this task, so item is left out of "combined" if present).

In [ ]:
VALENCE_CATEGORIES = [c for c in CATEGORIES if c != "item"]  # valence only applies to face/place trials

def plot_valence_by_category(data, title):
    per_subject = data.groupby(["subject", "object", "valence", "time_sec"])["value"].mean().reset_index()
    combined = (
        data[data["object"].isin(VALENCE_CATEGORIES)]
        .groupby(["subject", "valence", "time_sec"])["value"].mean().reset_index()
    )

    panels = VALENCE_CATEGORIES + ["combined"]
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.5), sharex=True, sharey=True)
    for ax, panel in zip(axes, panels):
        subset = combined if panel == "combined" else per_subject[per_subject["object"] == panel]
        sns.lineplot(data=subset, x="time_sec", y="value", hue="valence",
                     hue_order=["pos", "neg"], palette=VALENCE_COLORS, ax=ax)
        ax.axvline(TRIAL_ONSET_SEC, color="gray", linestyle="--", linewidth=0.75)
        ax.axvline(TRIAL_OFFSET_SEC, color="gray", linestyle=":", linewidth=0.75)
        ax.set_title(panel)
        ax.set_xlabel("Time (s)")
    axes[0].set_ylabel("classifier evidence (own true category)")
    fig.suptitle(title, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    return fig


for op in OPERATIONS:
    plot_valence_by_category(df[df["operation"] == op], f"{desc}: {op} -- pos vs. neg")
    plt.show()

## Plot 3: suppress/switch/clear (no maintain), one figure per valence

Single panel per figure -- one line per non-baseline operation, collapsed
across category, same `OPERATION_COLORS` as above.

In [ ]:
NON_BASELINE_OPERATIONS = [op for op in OPERATIONS if op != "maintain"]

def plot_operations_overlay(data, title):
    per_subject = (
        data[data["operation"].isin(NON_BASELINE_OPERATIONS)]
        .groupby(["subject", "operation", "time_sec"])["value"].mean().reset_index()
    )
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.lineplot(data=per_subject, x="time_sec", y="value", hue="operation",
                 hue_order=NON_BASELINE_OPERATIONS, palette=OPERATION_COLORS, ax=ax)
    ax.axvline(TRIAL_ONSET_SEC, color="gray", linestyle="--", linewidth=0.75)
    ax.axvline(TRIAL_OFFSET_SEC, color="gray", linestyle=":", linewidth=0.75)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("classifier evidence (own true category)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    return fig


for valence in ("pos", "neg"):
    plot_operations_overlay(df[df["valence"] == valence], f"{desc}: suppress/switch/clear -- {valence}")
    plt.show()

## Save outputs (optional)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

os.makedirs(OUTPUT_DIR, exist_ok=True)
pdf_path = os.path.join(OUTPUT_DIR, f"{desc}_category_evidence.pdf")

with PdfPages(pdf_path) as pdf:
    for valence in ("pos", "neg"):
        fig = plot_categories_by_operation(df[df["valence"] == valence], f"{desc}: category evidence by operation -- {valence}")
        pdf.savefig(fig)
        plt.close(fig)
    for op in OPERATIONS:
        fig = plot_valence_by_category(df[df["operation"] == op], f"{desc}: {op} -- pos vs. neg")
        pdf.savefig(fig)
        plt.close(fig)
    for valence in ("pos", "neg"):
        fig = plot_operations_overlay(df[df["valence"] == valence], f"{desc}: suppress/switch/clear -- {valence}")
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF written to: {pdf_path}")